# 07 · Federation → CLEAN via dbt (DuckLake × Trino)

**Pergunta:** dá pra o **dbt** ler **duas fontes** — uma tabela na nossa **DuckLake**
(RAW) e outra via **Trino** (federação, lendo um Postgres que aqui faz o papel do
Databricks/Snowflake do cliente) —, fazer um **JOIN** e materializar uma **nova
tabela CLEAN de volta na DuckLake**?

**Montagem (tudo local):**

```
Postgres (fed.products)  ──lido por──▶  Trino  ──┐
                                                 ├─▶  dbt-duckdb (JOIN)  ──▶  DuckLake (clean_order_lines)
DuckLake (orders = RAW)  ──attach nativo─────────┘
```

- **Engine do dbt = DuckDB** (`dbt-duckdb`). Lê a DuckLake nativamente (`ATTACH 'ducklake:'`).
- **Trino é a camada federada.** DuckDB não fala Trino nativo → a ponte é um **plugin
  dbt-duckdb** (cliente Trino → Arrow). Com isso cada tabela federada é declarada em
  **YAML** (`sources.yml`) e o join fica **SQL puro**. Em produção o Trino apontaria pro
  Databricks/Snowflake — muda só o `relation`/`catalog` no YAML.
- **Nada é copiado pra uma RAW intermediária** do lado federado: o dado do Trino entra
  em RAM, é joinado, e só a **CLEAN final** é gravada — na DuckLake.

> Requer Docker (imagem `trinodb/trino`) e o Postgres do starter no ar (`:5432`).

In [ ]:
import os, shutil, subprocess, sys, time
from pathlib import Path
import duckdb, trino

sys.path.insert(0, "/Users/allanfraga/Repos/strattum/experimentacoes")
import exp  # helpers da plataforma (REPO, dsn, attach_postgres)

BASE      = Path("/Users/allanfraga/Repos/strattum/experimentacoes/07-federation-dbt-clean")
OUT       = BASE / ".out"
CAT       = OUT / "lake" / "catalog.ducklake"   # catálogo DuckLake (nossa RAW/CLEAN)
DATA      = OUT / "lake" / "data"               # parquet da DuckLake
ENGINE_DB = OUT / "engine.duckdb"               # db "engine" do dbt-duckdb
DBT_DIR   = BASE / "dbt_fed"
TRINO_PORT = 8085
OUT.mkdir(parents=True, exist_ok=True)
print("duckdb", duckdb.__version__, "· paths ok")

## 1 · Sobe o Trino (engine de federação) com um catálogo do Postgres

O catálogo `postgresql` do Trino aponta pro Postgres do starter. É esse catálogo que,
em produção, seria trocado por `delta`/`snowflake` pra ler o lake do cliente.

In [ ]:
PW = next(l.split("=", 1)[1].strip()
          for l in open(f"{exp.REPO}/strattum-deploy/starter/.env")
          if l.startswith("POSTGRES_PASSWORD="))

cat_dir = OUT / "trino_catalog"; cat_dir.mkdir(parents=True, exist_ok=True)
(cat_dir / "postgresql.properties").write_text(
    "connector.name=postgresql\n"
    "connection-url=jdbc:postgresql://host.docker.internal:5432/demo_source\n"
    "connection-user=strattum\n"
    f"connection-password={PW}\n")

subprocess.run(["docker", "rm", "-f", "exp-trino"], capture_output=True)
subprocess.run(
    ["docker", "run", "-d", "--name", "exp-trino", "-p", f"{TRINO_PORT}:8080",
     "-v", f"{cat_dir}/postgresql.properties:/etc/trino/catalog/postgresql.properties:ro",
     "trinodb/trino:latest"], check=True, capture_output=True)

def trino_cur():
    return trino.dbapi.connect(host="localhost", port=TRINO_PORT, user="dbt",
                               catalog="postgresql", schema="fed").cursor()

print("aguardando Trino...")
for _ in range(60):
    try:
        cur = trino_cur(); cur.execute("SELECT 1"); cur.fetchall(); break
    except Exception:
        time.sleep(3)
else:
    raise RuntimeError("Trino não subiu a tempo")
print(f"Trino no ar em :{TRINO_PORT} (catálogo postgresql → Postgres)")

## 2 · Fonte FEDERADA: `products` no Postgres

Faz o papel do lake do cliente (Databricks/Snowflake). Semeada via DuckDB→Postgres só
pra deixar o notebook self-contained; o que importa é que o **Trino a lê por federação**.

In [ ]:
con = duckdb.connect()
pg = exp.attach_postgres(con, alias="pg", db="demo_source", read_only=False)
con.execute(f"CREATE SCHEMA IF NOT EXISTS {pg}.fed")
con.execute(f"DROP TABLE IF EXISTS {pg}.fed.products")
con.execute(f"""CREATE TABLE {pg}.fed.products AS SELECT * FROM (VALUES
    ('SKU-001','Bomba Centrifuga 5cv',  'Hidraulica',     4200.00),
    ('SKU-002','Valvula Esfera 2pol',   'Conexoes',        180.50),
    ('SKU-003','Motor Trifasico 10cv',  'Eletrica',       3100.00),
    ('SKU-004','Inversor de Frequencia','Eletrica',       2750.75),
    ('SKU-005','Sensor de Pressao',     'Instrumentacao',  640.00)
) t(sku, product_name, category, unit_price)""")
con.close()

cur = trino_cur(); cur.execute("SELECT count(*) FROM postgresql.fed.products")
print("products no Postgres, contados VIA TRINO (federação):", cur.fetchall()[0][0])

## 3 · Nossa RAW em DuckLake: `orders`

Tabela do nosso lake (DuckLake). É o outro lado do join.

In [ ]:
shutil.rmtree(OUT / "lake", ignore_errors=True)
ENGINE_DB.unlink(missing_ok=True)
DATA.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute("INSTALL ducklake; LOAD ducklake")
con.execute(f"ATTACH 'ducklake:{CAT}' AS lake (DATA_PATH '{DATA}/')")
con.execute("""CREATE TABLE lake.orders AS SELECT * FROM (VALUES
    (1,'SKU-001','ana@acme.com',     2, TIMESTAMP '2026-07-01 09:00'),
    (2,'SKU-003','ana@acme.com',     1, TIMESTAMP '2026-07-02 10:00'),
    (3,'SKU-002','joao@beta.com',   10, TIMESTAMP '2026-07-02 11:00'),
    (4,'SKU-005','joao@beta.com',    4, TIMESTAMP '2026-07-03 08:00'),
    (5,'SKU-004','maria@gama.com',   1, TIMESTAMP '2026-07-03 14:00'),
    (6,'SKU-001','maria@gama.com',   1, TIMESTAMP '2026-07-04 16:00'),
    (7,'SKU-002','ana@acme.com',     5, TIMESTAMP '2026-07-05 09:30'),
    (8,'SKU-003','carlos@delta.com', 2, TIMESTAMP '2026-07-05 10:15')
) t(order_id, product_sku, customer_email, qty, order_ts)""")
print("DuckLake orders:", con.execute("SELECT count(*) FROM lake.orders").fetchone()[0], "linhas")
con.execute("DETACH lake"); con.close()

## 4 · O que o dbt faz — 3 peças

- **`plugins/trino_source.py`** — plugin dbt-duckdb que serve **qualquer** tabela federada
  do Trino (a ponte DuckDB↔Trino; schema Arrow **inferido**, nada hardcoded).
- **`models/sources.yml`** — declara as duas fontes: `raw_lake.orders` (DuckLake) e
  `federated.products` (Trino, via o plugin). Tabela federada nova = só mais um item aqui.
- **`models/clean_order_lines.sql`** — o **JOIN** em SQL puro; `database='lake'`
  materializa na DuckLake.

In [ ]:
for f in ["plugins/trino_source.py", "models/sources.yml", "models/clean_order_lines.sql"]:
    print(f"\n===== dbt_fed/{f} " + "=" * (46 - len(f)))
    print((DBT_DIR / f).read_text().rstrip())

## 5 · `dbt build`

Paths e host do Trino entram por env var (o `profiles.yml` usa `env_var(...)`).

In [ ]:
env = {**os.environ,
       "FED_ENGINE_DB": str(ENGINE_DB), "DUCKLAKE_CAT": str(CAT),
       "DUCKLAKE_DATA": str(DATA), "DBT_PROFILES_DIR": str(DBT_DIR),
       "PYTHONPATH": str(DBT_DIR),  # pro `plugins.trino_source` importar
       "TRINO_HOST": "localhost", "TRINO_PORT": str(TRINO_PORT)}

dbt_bin = str(BASE.parent / ".venv" / "bin" / "dbt")
r = subprocess.run([dbt_bin, "build"], cwd=DBT_DIR, env=env,
                   capture_output=True, text=True)
print(r.stdout)
assert r.returncode == 0, r.stderr

## 6 · Verifica a CLEAN materializada na DuckLake

`clean_order_lines` foi criada **na DuckLake** (banco `lake`), juntando `qty`/e-mail
(nossa RAW) com `product_name`/`category`/`unit_price` (fonte federada via Trino).
`line_total = qty × unit_price` foi computado no join.

In [ ]:
con = duckdb.connect()
con.execute("INSTALL ducklake; LOAD ducklake")
con.execute(f"ATTACH 'ducklake:{CAT}' AS lake (DATA_PATH '{DATA}/', READ_ONLY)")
rel = con.sql("""
    SELECT order_id, customer_email, product_name, category,
           qty, unit_price, line_total
    FROM lake.clean_order_lines ORDER BY order_id""")
rel.show()
n = con.execute("SELECT count(*) FROM lake.clean_order_lines").fetchone()[0]
print(f"✅ {n} linhas na CLEAN (lake.clean_order_lines), materializadas em DuckLake")
con.close()

## Conclusão

**Sim — funciona.** O `dbt` (engine **DuckDB**) leu a `orders` da **DuckLake** (attach
nativo) + a `products` **federada via Trino**, fez o **JOIN** e gravou `clean_order_lines`
**de volta na DuckLake**. Um único `dbt build`, dois modelos.

**Como o Trino entra:** o DuckDB **não tem connector Trino nativo** — a ponte é um
**plugin dbt-duckdb** usando o **cliente Trino (DBAPI)** (alternativa direta ao ODBC/JDBC;
o DBAPI é o mais simples em Python). O plugin serve **qualquer** tabela federada: adicionar
uma fonte = mais um item no `sources.yml`, sem Python novo. Em produção troca-se só o
`relation`/`catalog` no YAML (`postgresql` → `delta`/`snowflake`).

Detalhes, alternativas (plugin dbt-duckdb) e caveats: [`RESULTADOS.md`](RESULTADOS.md).

Teardown do Trino abaixo (opcional).

In [ ]:
subprocess.run(["docker", "rm", "-f", "exp-trino"], capture_output=True)
print("exp-trino removido")